In [2]:
import ee
import sys
import importlib
import geopandas as gpd
sys.path.insert(0, '/Users/pstano/Documents/applications/lucera')

import utilities
importlib.reload(utilities)

from utilities import (
    DatasetBuilder, 
    H3GridBuilder,
    KroneckerGridBuilder,
    AlphaEarth, 
    ESAWorldCover
)

In [6]:
ee.Initialize(project="lucera-494618")

nuts = gpd.read_file("NUTS_RG_01M_2021_4326.gpkg")
country_bbox = nuts[nuts['CNTR_CODE'] == 'PL'].union_all()

pl2 = nuts[nuts['NUTS_ID'].isin(['PL21', 'PL22'])].union_all()
PL2_BBOX = pl2.bounds

KR_FILENAME = 'kr_pl_worldcover_2023'
H3_FILENAME = 'h3_pl_worldcover_2023'

In [3]:
h3_grid = H3GridBuilder(bbox=country_bbox.bounds, resolution=7)
kr_grid = KroneckerGridBuilder(bbox=country_bbox.bounds, n_points=len(h3_grid))

In [4]:
dataset_homogeneous = (
    DatasetBuilder()
    .from_grid(h3_grid)
    .set_label_source(ESAWorldCover())
    .add_source(AlphaEarth(year=2023))
    .build(strategy='homogeneous', threshold=0.8)
    .save(f"data/{H3_FILENAME}_homogeneous.csv")
)

Building homogeneous dataset (threshold=0.8)...


Computing homogeneity: 100%|██████████| 940/940 [18:17<00:00,  1.17s/it]


Kept 24027/93991 cells above threshold 0.8


Fetching AlphaEarth embeddings: 100%|██████████| 49/49 [10:59<00:00, 13.45s/it]


Dataset built — 24027 rows, 69 columns
Saved to data/h3_pl_worldcover_2023_homogeneous.csv


In [ ]:
dataset_adaptive = (
    DatasetBuilder()
    .from_grid(H3GridBuilder(bbox=PL2_BBOX, resolution=7))
    .set_label_source(ESAWorldCover())
    .add_source(AlphaEarth(year=2023))
    .build(strategy='adaptive', threshold=0.8, max_depth=2, checkpoint_dir='data/pl2_checkpoints')
    .save("data/pl2_adaptive.csv")
)

Building adaptive dataset (threshold=0.8, max_depth=2)...
Depth 0: processing 93991 cells at res 7


Computing homogeneity: 100%|██████████| 940/940 [15:38<00:00,  1.00it/s]   


  Clean: 24027, Noisy: 68621
Depth 1: processing 480347 cells at res 8


Computing homogeneity: 100%|██████████| 4804/4804 [1:56:18<00:00,  1.45s/it]    


  Clean: 118821, Noisy: 361526
Depth 2: processing 2530682 cells at res 9


Computing homogeneity:  58%|█████▊    | 14754/25307 [8:11:40<5:51:40,  2.00s/it]  


KeyboardInterrupt: 

In [6]:
from utilities.region_manager import RegionIndex
from pathlib import Path

idx = RegionIndex(path='/Users/pstano/Documents/applications/lucera/data/index.parquet')

failed = idx.failed()
running = idx.running()
print(f"Failed regions: {len(failed)}")
print(f"Running regions: {len(running)}")

for _, region in failed.iterrows():
    idx.update(region['region_id'], status='pending')

for _, region in running.iterrows():
    idx.update(region['region_id'], status='pending')

print("Reset done")
idx.summary()

Failed regions: 0
Running regions: 1
Reset done
Total regions: 808
status
complete    806
pending       2

Countries: ['PL', 'PT', 'IS']
